# EWS Data Indexing and Coverage Analysis

This notebook indexes the downloaded IDX files from Google Drive (`G:\My Drive\Dataset PUI-PT\IDX_Downloads`) and matches them against the listing of valid companies from the tracking spreadsheet. 

The analysis calculates overall data coverage, identifies missing files (Laporan Keuangan or Laporan Tahunan) for each year, and lists the companies that have complete data across all 4 years (2021-2024).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Configure styling
sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
# Config & Paths
excel_path = r"../Data_Crawling/Rekap_Perusahaan_2021_2024.xlsx"
downloads_dir = r"G:\My Drive\Dataset PUI-PT\IDX_Downloads"
years = [2021, 2022, 2023, 2024]
categories = ["Laporan_Keuangan", "Laporan_Tahunan", "Dokumen_Lainnya"]

In [ ]:
# 1. Load tracking sheet
df_companies = pd.read_excel(excel_path)
df_companies.columns = [c.strip() for c in df_companies.columns]

# Filter for Valid companies
df_valid = df_companies[df_companies['Status'].astype(str).str.strip().str.lower() == 'valid'].copy()
df_valid['Kode'] = df_valid['Kode'].astype(str).str.strip()

print(f"Total Perusahaan di Sheet: {len(df_companies)}")
print(f"Total Perusahaan Valid: {len(df_valid)}")

valid_tickers = sorted(list(set(df_valid['Kode'])))

In [ ]:
# 2. Scan downloads_dir using optimized direct path checking
index_data = []

if not os.path.exists(downloads_dir):
    print(f"ERROR: Folder {downloads_dir} tidak ditemukan!")
else:
    print("Memulai scanning direktori...")
    for ticker in tqdm(valid_tickers, desc="Scanning Emiten"):
        ticker_path = os.path.join(downloads_dir, ticker)
        if not os.path.exists(ticker_path):
            continue
            
        for year in years:
            year_path = os.path.join(ticker_path, str(year))
            if not os.path.exists(year_path):
                continue
                
            for category in categories:
                category_path = os.path.join(year_path, category)
                if not os.path.exists(category_path):
                    continue
                    
                try:
                    files = os.listdir(category_path)
                except Exception as e:
                    continue
                    
                pdf_files = [f for f in files if f.lower().endswith('.pdf')]
                
                for f in pdf_files:
                    try:
                        f_size = os.path.getsize(os.path.join(category_path, f))
                    except:
                        f_size = 0
                    if f_size > 1000:
                        index_data.append({
                            "Kode": ticker,
                            "Tahun": year,
                            "Kategori": category,
                            "File": f
                        })

df_index = pd.DataFrame(index_data)
print(f"Total file PDF berhasil diindeks: {len(df_index)}")

In [ ]:
# 3. Compute presence matrix
records = []
for ticker in valid_tickers:
    matching_row = df_valid[df_valid['Kode'] == ticker]
    if matching_row.empty:
        continue
    name = matching_row['Nama Perusahaan'].values[0]
    for year in years:
        records.append({
            "Kode": ticker,
            "Nama Perusahaan": name,
            "Tahun": year,
            "Laporan_Keuangan_Exists": False,
            "Laporan_Tahunan_Exists": False,
            "Lengkap_Tahun_Ini": False
        })
        
df_coverage = pd.DataFrame(records)
df_coverage.set_index(["Kode", "Tahun"], inplace=True)

if not df_index.empty:
    for idx, row in df_index.iterrows():
        key = (row['Kode'], row['Tahun'])
        if key in df_coverage.index:
            if row['Kategori'] == 'Laporan_Keuangan':
                df_coverage.loc[key, 'Laporan_Keuangan_Exists'] = True
            elif row['Kategori'] == 'Laporan_Tahunan':
                df_coverage.loc[key, 'Laporan_Tahunan_Exists'] = True

df_coverage['Lengkap_Tahun_Ini'] = df_coverage['Laporan_Keuangan_Exists'] & df_coverage['Laporan_Tahunan_Exists']
df_coverage.reset_index(inplace=True)

total_combinations = len(df_coverage)
lk_count = df_coverage['Laporan_Keuangan_Exists'].sum()
lt_count = df_coverage['Laporan_Tahunan_Exists'].sum()
complete_count = df_coverage['Lengkap_Tahun_Ini'].sum()

print("SUMMARY COVERAGE DATA:")
print(f"Total Emiten x Tahun (Expected): {total_combinations} ({len(valid_tickers)} emiten x {len(years)} tahun)")
print(f"Laporan Keuangan Tersedia: {lk_count} ({lk_count / total_combinations * 100:.2f}%)")
print(f"Laporan Tahunan Tersedia:  {lt_count} ({lt_count / total_combinations * 100:.2f}%)")
print(f"Lengkap Keduanya (LK & LT): {complete_count} ({complete_count / total_combinations * 100:.2f}%)")

In [ ]:
# 4. Export Missing Files Report
df_missing = df_coverage[~df_coverage['Lengkap_Tahun_Ini']].copy()
conditions = []
for idx, row in df_missing.iterrows():
    missing_items = []
    if not row['Laporan_Keuangan_Exists']:
        missing_items.append("Laporan Keuangan")
    if not row['Laporan_Tahunan_Exists']:
        missing_items.append("Laporan Tahunan")
    conditions.append(", ".join(missing_items))
    
df_missing['Keterangan Hilang'] = conditions

df_missing_report = df_missing[["Kode", "Nama Perusahaan", "Tahun", "Laporan_Keuangan_Exists", "Laporan_Tahunan_Exists", "Keterangan Hilang"]].copy()
df_missing_report.columns = ["Kode Emiten", "Nama Perusahaan", "Tahun", "Ada Laporan Keuangan", "Ada Laporan Tahunan", "Keterangan Hilang"]

df_missing_report.to_excel("Missing_Files_Report.xlsx", index=False)
print(f"Missing files exported to 'Missing_Files_Report.xlsx' ({len(df_missing_report)} rows)")

In [ ]:
# 5. Export Complete Companies Report
complete_by_year = df_coverage.groupby("Kode")["Lengkap_Tahun_Ini"].all()
df_complete_companies = df_valid[df_valid['Kode'].isin(complete_by_year[complete_by_year].index)].copy()

df_summary_4yr = df_coverage.pivot(index="Kode", columns="Tahun", values="Lengkap_Tahun_Ini")
df_summary_4yr.columns = [f"Lengkap_{y}" for y in df_summary_4yr.columns]

df_complete_report = df_complete_companies.merge(df_summary_4yr, left_on="Kode", right_index=True)
df_complete_report = df_complete_report[["Kode", "Nama Perusahaan", "Status", "Lengkap_2021", "Lengkap_2022", "Lengkap_2023", "Lengkap_2024"]]
df_complete_report.to_excel("Complete_Companies_Report.xlsx", index=False)
print(f"Complete companies exported to 'Complete_Companies_Report.xlsx' ({len(df_complete_report)} rows)")

In [ ]:
# 6. Visualizations
plt.figure(figsize=(10, 6))
df_yearly_summary = df_coverage.groupby("Tahun").agg({
    "Laporan_Keuangan_Exists": "mean",
    "Laporan_Tahunan_Exists": "mean",
    "Lengkap_Tahun_Ini": "mean"
}) * 100

df_yearly_summary_melted = df_yearly_summary.reset_index().melt(id_vars="Tahun", var_name="Kategori", value_name="Persentase")
df_yearly_summary_melted["Kategori"] = df_yearly_summary_melted["Kategori"].map({
    "Laporan_Keuangan_Exists": "Laporan Keuangan",
    "Laporan_Tahunan_Exists": "Laporan Tahunan",
    "Lengkap_Tahun_Ini": "Lengkap Keduanya"
})

ax = sns.barplot(data=df_yearly_summary_melted, x="Tahun", y="Persentase", hue="Kategori", palette="viridis")
plt.title("Persentase Cakupan (Coverage) Dokumen IDX per Tahun", fontsize=14, pad=15)
plt.ylabel("Cakupan (%)", fontsize=12)
plt.xlabel("Tahun", fontsize=12)
plt.ylim(0, 110)

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.1f}%',
                    xy=(p.get_x() + p.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("coverage_chart.png", dpi=150)
plt.show()